# C2-linear-models — Practice p20

**Type:** constrained coding · **Difficulty:** core · **Concepts:** linear-regression-estimator-derivation

Implement \`ols_full_rank(X, y)\`; an intercept is already a column of
$X$.

Set \`COND_MAX = 1e8\`. Accept finite numeric \`X (n, p)\` and
\`y (n,)\` only when:

- $n\ge p$ and row counts match;
- \`np.linalg.matrix_rank(X) == p\`;
- \`np.linalg.cond(X.T @ X) <= COND_MAX\`.

Raise \`ValueError\` before solving for every other shape/domain case,
including rank-deficient and full-rank-but-ill-conditioned designs.
Do not mutate inputs.

On accepted input return finite float \`beta (p,)\` satisfying the normal
system and $X^T(X\beta-y)=0$ at
\`ATOL = 1e-10\`, \`RTOL = 0.0\`.

**Required route:** exactly one \`np.linalg.solve(X.T @ X, X.T @ y)\`
call per accepted invocation and zero solve calls on rejected input.

**Banned inside the function (zero points):** \`np.linalg.inv\`,
\`np.linalg.pinv\`, \`np.linalg.lstsq\`, any spelling of
\`sklearn\` or \`statsmodels\`, loops, comprehensions, and recursion.
Do not hardcode the visible examples; the immutable checker uses
secondary fixtures and adversarial inputs.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0
COND_MAX = 1e8


def ols_full_rank(X, y):
    # YOUR CODE HERE
    ...

## Immutable contract check — do not edit

The checker independently computes reference coefficients on two
well-conditioned fixtures, checks coefficients/residuals/orthogonality,
exercises malformed, rank-deficient, and numerically ill-conditioned
inputs, audits nested bytecode/source for banned routes and loops, and
instruments \`np.linalg.solve\` to require exactly one call on success
and none before \`ValueError\`.

In [ ]:
import dis
import inspect
import types

_ORIGINAL_SOLVE_P20 = np.linalg.solve
_ORIGINAL_LSTSQ_P20 = np.linalg.lstsq


def _nested_code_p20(fn):
    pending = [fn.__code__]
    found = []
    while pending:
        current = pending.pop()
        found.append(current)
        pending.extend(
            item for item in current.co_consts
            if isinstance(item, types.CodeType)
        )
    return found


_codes_p20 = _nested_code_p20(ols_full_rank)
_banned_names_p20 = {
    "inv", "pinv", "lstsq", "sklearn", "statsmodels",
}
for _code_p20 in _codes_p20:
    _names_p20 = {name.lower() for name in _code_p20.co_names}
    assert not (_names_p20 & _banned_names_p20)
    assert ols_full_rank.__name__ not in _code_p20.co_names
    _ops_p20 = {item.opname for item in dis.get_instructions(_code_p20)}
    assert "FOR_ITER" not in _ops_p20
    assert not any(name.startswith("JUMP_BACKWARD") for name in _ops_p20)

try:
    _source_p20 = inspect.getsource(ols_full_rank).lower()
except (OSError, TypeError):
    _source_p20 = ""
assert all(token not in _source_p20 for token in (
    "np.linalg.inv(", "np.linalg.pinv(", "np.linalg.lstsq(",
    "sklearn", "statsmodels",
))

_X1_p20 = np.array([
    [1.0, -2.0, 0.0],
    [1.0, 0.0, 1.0],
    [1.0, 1.0, -1.0],
    [1.0, 3.0, 2.0],
    [1.0, 4.0, 0.0],
])
_y1_p20 = np.array([-0.2, 0.8, 2.2, 6.4, 6.3])
_X2_p20 = np.array([
    [1.0, -3.0],
    [1.0, -1.0],
    [1.0, 2.0],
    [1.0, 4.0],
    [1.0, 6.0],
    [1.0, 9.0],
])
_y2_p20 = np.array([-4.0, -0.5, 4.2, 7.1, 10.5, 14.8])
_accepted_p20 = ((_X1_p20, _y1_p20), (_X2_p20, _y2_p20))

for _X_p20, _y_p20 in _accepted_p20:
    _expected_p20 = _ORIGINAL_LSTSQ_P20(_X_p20, _y_p20, rcond=None)[0]
    _X_before_p20 = _X_p20.copy()
    _y_before_p20 = _y_p20.copy()
    _solve_calls_p20 = []

    def _counted_solve_p20(a, b):
        _solve_calls_p20.append((np.array(a, copy=True), np.array(b, copy=True)))
        return _ORIGINAL_SOLVE_P20(a, b)

    np.linalg.solve = _counted_solve_p20
    try:
        _beta_p20 = ols_full_rank(_X_p20, _y_p20)
    finally:
        np.linalg.solve = _ORIGINAL_SOLVE_P20

    assert len(_solve_calls_p20) == 1
    assert isinstance(_beta_p20, np.ndarray)
    assert _beta_p20.shape == (_X_p20.shape[1],)
    assert np.issubdtype(_beta_p20.dtype, np.floating)
    assert np.isfinite(_beta_p20).all()
    assert np.array_equal(_X_p20, _X_before_p20)
    assert np.array_equal(_y_p20, _y_before_p20)
    assert np.allclose(_beta_p20, _expected_p20, atol=ATOL, rtol=RTOL)
    _resid_p20 = _X_p20 @ _beta_p20 - _y_p20
    _expected_resid_p20 = _X_p20 @ _expected_p20 - _y_p20
    assert np.allclose(
        _resid_p20, _expected_resid_p20, atol=ATOL, rtol=RTOL,
    )
    assert np.allclose(
        _X_p20.T @ _X_p20 @ _beta_p20,
        _X_p20.T @ _y_p20,
        atol=ATOL, rtol=RTOL,
    )
    assert np.allclose(
        _X_p20.T @ _resid_p20,
        np.zeros(_X_p20.shape[1]),
        atol=ATOL, rtol=RTOL,
    )

_X_ill_p20 = np.array([
    [1.0, 1.0],
    [1.0, 1.0001],
    [1.0, 0.9999],
    [1.0, 1.0002],
])
assert np.linalg.matrix_rank(_X_ill_p20) == 2
assert np.linalg.cond(_X_ill_p20.T @ _X_ill_p20) > COND_MAX

_rejected_p20 = (
    (np.ones(3), np.ones(3)),
    (np.ones((3, 2)), np.ones((3, 1))),
    (np.ones((3, 2)), np.ones(2)),
    (np.ones((2, 3)), np.ones(2)),
    (np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]]), np.ones(3)),
    (_X_ill_p20, np.arange(4.0)),
    (np.array([[1.0, np.nan], [1.0, 2.0]]), np.ones(2)),
)
for _X_bad_p20, _y_bad_p20 in _rejected_p20:
    _solve_calls_p20 = []

    def _reject_counted_solve_p20(a, b):
        _solve_calls_p20.append((a, b))
        return _ORIGINAL_SOLVE_P20(a, b)

    np.linalg.solve = _reject_counted_solve_p20
    try:
        try:
            ols_full_rank(_X_bad_p20, _y_bad_p20)
        except ValueError:
            pass
        else:
            raise AssertionError("rejected input must raise ValueError")
    finally:
        np.linalg.solve = _ORIGINAL_SOLVE_P20
    assert _solve_calls_p20 == []